In [1]:
# 1. Import library
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

import pickle
import matplotlib.pyplot as plt

In [2]:

data_dir = "../dataset/food-20"  # path dataset
img_height, img_width = 224, 224
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/train",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/val",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names

# Optimize dataset (prefetching)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 4. Base model MobileNetV2 (transfer learning)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze weights

# 5. Add custom classifier on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

# 6. Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 7. Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  # bisa ditambah kalau GPU kuat
)

# 8. Cek performa
acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"Training Accuracy: {acc:.2f}, Validation Accuracy: {val_acc:.2f}")

# 9. Save model
model.save("mobilenetv2.h5")
print("✅ Model MobileNetV2 saved as mobilenetv2.h5")

Found 1400 files belonging to 20 classes.
Found 400 files belonging to 20 classes.
Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 39s 748ms/step - accuracy: 0.0627 - loss: 3.4140 - val_accuracy: 0.1400 - val_loss: 2.8554
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 32s 720ms/step - accuracy: 0.1179 - loss: 2.9975 - val_accuracy: 0.1875 - val_loss: 2.7037
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 31s 715ms/step - accuracy: 0.1679 - loss: 2.7371 - val_accuracy: 0.2200 - val_loss: 2.5974
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 31s 699ms/step - accuracy: 0.1954 - loss: 2.6124 - val_accuracy: 0.2150 - val_loss: 2.5333
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 31s 698ms/step - accuracy: 0.2405 - loss: 2.5038 - val_accuracy: 0.2275 - val_loss: 2.5224
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 30s 690ms/step - accuracy: 0.2441 - loss: 2.4168 - val_accuracy: 0.2425 - val_loss: 2.5034
Epoch 7/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 31s 713ms/step - accuracy: 0.3039 - loss: 2.3036 - val_accuracy: 0.2225 - val_loss: 2.5200
Epoch 8/10
44/

Training Accuracy: 0.35, Validation Accuracy: 0.24
✅ Model MobileNetV2 saved as mobilenetv2.h5


In [3]:
print("Jumlah class:", len(class_names))

count_train = sum(1 for _ in train_ds.unbatch())
count_val = sum(1 for _ in val_ds.unbatch())

print("Jumlah gambar training:", count_train)
print("Jumlah gambar validation:", count_val)
print("Class names:", class_names)


Jumlah class: 20
Jumlah gambar training: 1400
Jumlah gambar validation: 400
Class names: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'breakfast_burrito', 'caesar_salad', 'cheesecake', 'chicken_curry', 'churros', 'donuts', 'edamame', 'falafel', 'french_fries', 'frozen_yogurt', 'grilled_cheese_sandwich', 'hamburger', 'hot_and_sour_soup', 'ice_cream', 'lasagna']
